# Evaluation A — Canonical Error Inspection

This concise, post-hoc workspace filters the canonical one-row-per-case evaluator artifacts. It does not reconstruct predictions or recompute any metric. A mismatch with a SHERLOC silver-reference label is not automatically a factual error; later human adjudication is a separate Evaluation B activity.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, SVG, display


def locate_repo_root() -> Path:
    configured = os.environ.get("SHERLOC_REPO_ROOT")
    starts = [Path(configured).expanduser()] if configured else []
    starts.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in starts:
        if (candidate / "src/experiments/11_evaluate_amp.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate SHERLOC_Case_Analysis. Start Jupyter in the repository "
        "or set SHERLOC_REPO_ROOT."
    )


REPO_ROOT = locate_repo_root()
ANALYSIS_ROOT = REPO_ROOT / "outputs/analysis/evaluation_a"
FIGURE_ROOT = REPO_ROOT / "outputs/figures/evaluation_a"
METRICS_ROOT = REPO_ROOT / "outputs/metrics"


def load_csv(path: Path, required_columns=()) -> pd.DataFrame:
    """Load a finalized artifact without synthesizing missing rows."""
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{path.relative_to(REPO_ROOT)}`"))
        return pd.DataFrame(columns=list(required_columns))
    frame = pd.read_csv(path)
    missing = set(required_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    return frame


def load_json(path: Path) -> dict:
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{path.relative_to(REPO_ROOT)}`"))
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def show_table(frame: pd.DataFrame, *, empty_message="No finalized rows are available."):
    if frame.empty:
        display(Markdown(f"> **NOT YET AVAILABLE:** {empty_message}"))
    else:
        display(frame)


def show_figure(filename: str):
    """Display a finalized SVG; never recreate a figure in the notebook."""
    path = FIGURE_ROOT / filename
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{path.relative_to(REPO_ROOT)}`"))
        return
    display(SVG(filename=str(path)))


manifest = load_json(METRICS_ROOT / "amp_evaluation_manifest.json")
completion_gate = manifest.get("final_completion_gate", "NOT YET AVAILABLE")
display(Markdown(f"**Canonical Evaluation A completion gate:** `{completion_gate}`"))


## 2. Load canonical case-level rows

In [ ]:
error_columns = (
    "method", "case_id", "search_rank", "jurisdiction", "fold", "fact_summary",
    "silver_reference_amp_json", "predicted_amp_json", "false_positive_labels_json",
    "false_negative_labels_json", "exact_set_correct", "example_jaccard",
    "truncated_input", "act_cpmr", "act_contained_recall", "means_cpmr",
    "means_contained_recall", "purpose_cpmr", "purpose_contained_recall",
)
a1_errors = load_csv(METRICS_ROOT / "a1/amp_case_level_errors.csv", error_columns)
a2_errors = load_csv(METRICS_ROOT / "a2/amp_case_level_errors.csv", error_columns)
if not a1_errors.empty:
    a1_errors = a1_errors.assign(evaluation="A1")
if not a2_errors.empty:
    a2_errors = a2_errors.assign(evaluation="A2")
case_errors = pd.concat([a1_errors, a2_errors], ignore_index=True)
show_table(case_errors.head(20))


## 3. Reproducible inspection filters

In [ ]:
# Record these values with every exported or cited case set.
EVALUATION = "A2"       # "A1", "A2", or None
METHOD = None           # "M1", "M2", "M3", "M4", or None
FOLD = None             # 1, 2, 3, or None
JURISDICTION = None     # exact string or None
LABEL_ID = None         # exact frozen ontology ID or None
EXACT_SET_FAILURES_ONLY = True
CPMR_FAMILY = None      # "act", "means", "purpose", or None
CPMR_SUCCESS_ONLY = False
MAX_ROWS = 100

filtered = case_errors.copy()
if EVALUATION is not None:
    filtered = filtered.loc[filtered["evaluation"].eq(EVALUATION)]
if METHOD is not None:
    filtered = filtered.loc[filtered["method"].eq(METHOD)]
if FOLD is not None:
    filtered = filtered.loc[filtered["fold"].eq(FOLD)]
if JURISDICTION is not None:
    filtered = filtered.loc[filtered["jurisdiction"].eq(JURISDICTION)]
if EXACT_SET_FAILURES_ONLY:
    filtered = filtered.loc[filtered["exact_set_correct"].eq(0)]
if LABEL_ID is not None:
    def contains_label(value):
        return LABEL_ID in json.loads(value)
    filtered = filtered.loc[
        filtered["false_positive_labels_json"].map(contains_label)
        | filtered["false_negative_labels_json"].map(contains_label)
    ]
if CPMR_FAMILY not in (None, "act", "means", "purpose"):
    raise ValueError("CPMR_FAMILY must be act, means, purpose, or None")
if CPMR_SUCCESS_ONLY:
    if CPMR_FAMILY is None:
        raise ValueError("Set CPMR_FAMILY before requesting CPMR successes")
    filtered = filtered.loc[filtered[f"{CPMR_FAMILY}_cpmr"].eq(1)]

inspection_columns = [
    "evaluation", "method", "search_rank", "jurisdiction", "fold",
    "exact_set_correct", "example_jaccard", "truncated_input",
    "silver_reference_amp_json", "predicted_amp_json",
    "false_positive_labels_json", "false_negative_labels_json",
    "act_cpmr", "act_contained_recall", "means_cpmr", "means_contained_recall",
    "purpose_cpmr", "purpose_contained_recall", "fact_summary",
]
show_table(filtered.sort_values(["evaluation", "search_rank"])[inspection_columns].head(MAX_ROWS))


## 4. Finalized sensitivity and M3/M4 context

In [ ]:
rare_sensitivity = load_csv(
    ANALYSIS_ROOT / "rare_label_sensitivity.csv", ("evaluation", "method")
)
m3_m4_per_label = load_csv(
    ANALYSIS_ROOT / "m3_vs_m4_per_label_f1.csv", ("evaluation", "label_id")
)
show_table(rare_sensitivity)
show_table(m3_m4_per_label)


## 5. Core diagnostic figures

In [ ]:
show_figure("figure_3_cpmr_vs_contained_recall.svg")
show_figure("figure_4_per_label_f1.svg")


## 6. Interpretation boundary

Any case selection developed after seeing benchmark outcomes is **post-hoc exploratory analysis** and must be reported as such. Do not use this notebook to tune or rerun Evaluation A, select human reliability cases, reveal hidden reviewer material, or adjudicate silver/human disagreements.